# تدريب أدوات الورشة على GPU — YOLO

1. من **Runtime → Change runtime type** اختر **T4 GPU** أو أي GPU متاح ثم Save.
2. اختر **Runtime → Run all**.
3. وافق على ربط Google Drive عندما يطلب Colab ذلك؛ تحفظ الأوزان والنتائج في مجلد `WoodWorkShop_runs`.

هذا الدفتر يدرب نموذجاً جديداً على البيانات الكاملة: أقلام (بما فيها أقلام الرصاص)، مفكات، وبراغي. صور الخلفية تبقى بدون ملفات txt. يقيّم على validation، ويعيد التدريب إذا لزم، ثم يختبر النموذج المختار. لا يحوّل جلسة CPU الحالية إلى GPU ولا يستأنفها تلقائياً.

توفر GPU ومدة الجلسة يعتمدان على حساب Colab. إذا انقطعت الجلسة، تبقى نقاط الحفظ المكتوبة في Drive. تجنّب تشغيل جلستين على مجلد نتائج واحد.


In [ ]:
import sys, subprocess
from pathlib import Path
import torch

if not torch.cuda.is_available():
    raise RuntimeError("اختر GPU من Runtime > Change runtime type ثم أعد تشغيل الخلايا.")
print("GPU:", torch.cuda.get_device_name(0))
print("PyTorch:", torch.__version__)
repo = Path('/content/WoodWorkShop')
if not repo.exists():
    subprocess.check_call(['git', 'clone', '--depth', '1', 'https://github.com/omar-salem23/WoodWorkShop.git', str(repo)])
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', '-r', str(repo / 'requirements.txt')])
print('Repository:', repo)


## حفظ النتائج في Google Drive

ستطلب هذه الخلية تفويضك للوصول إلى Drive. سيُنشأ مجلد جديد لكل تشغيل كامل، وتُحفظ نقاط الحفظ بعد كل epoch. بيانات الصور تُنزّل إلى قرص Colab لتسريع القراءة.


In [ ]:
from google.colab import drive
from datetime import datetime, timezone
import os

drive.mount('/content/drive')
run_name = datetime.now(timezone.utc).strftime('gpu_%Y%m%d_%H%M%S_UTC')
output = Path('/content/drive/MyDrive/WoodWorkShop_runs') / run_name
output.mkdir(parents=True, exist_ok=False)
work = Path('/content/woodwork_data')
work.mkdir(parents=True, exist_ok=True)
os.environ['YOLO_CONFIG_DIR'] = '/content/ultralytics_settings'
print('Results and checkpoints:', output)


## تنزيل البيانات وتجهيزها

الأرشيف الأصلي لا يتغير. نستبعد النسخ المتطابقة من النسخة المعدة ونصلح أخطاء الصناديق المسجلة في التقرير. صور الخلفية المعتمدة تبقى بلا تسميات؛ لا نضيف فئة رابعة.


In [ ]:
import zipfile
from pathlib import PurePosixPath
import gdown

archive = work / 'dataset.zip'
if not archive.exists() or not zipfile.is_zipfile(archive):
    result = gdown.download(id='1vz95eqHeMBBZoNrfANFlW_Mzv_Tf6LE6', output=str(archive), quiet=False)
    if result is None:
        raise RuntimeError('تعذر تنزيل الملف من Drive. تحقق من المشاركة أو حصة التنزيل.')
source = work / 'source'
if not (source / '.extracted').exists():
    with zipfile.ZipFile(archive) as z:
        for item in z.infolist():
            path = PurePosixPath(item.filename)
            if path.is_absolute() or '..' in path.parts or '\\' in item.filename:
                raise ValueError('Unsafe ZIP member')
        if sum(i.file_size for i in z.infolist()) > 8 * 1024**3:
            raise ValueError('Unexpected archive size')
        z.extractall(source)
    (source / '.extracted').touch()
prepared = work / 'prepared'
if not (prepared / 'preparation_report.json').exists():
    subprocess.check_call([sys.executable, str(repo / 'prepare_dataset.py'),
        str(source / 'dataset_split/data.yaml'), '--output', str(prepared),
        '--missing-labels', 'background'])
subprocess.check_call([sys.executable, str(repo / 'audit_dataset.py'), str(prepared)])
import shutil
shutil.copyfile(prepared / 'preparation_report.json', output / 'preparation_report.json')


## التدريب والتقييم

الإعدادات: YOLO11n بأوزان جاهزة، حجم صورة 640، batch 16، و40 epoch أولاً، ثم حتى 60 إضافية إذا كانت نتائج التحقق أقل من الأهداف. التوقف المبكر قد ينهي المرحلة قبل ذلك.

يقيس التقرير mAP لكل فئة ونسبة صور الخلفية التي ظهر عليها مربع خاطئ بثقة 0.25 أو أعلى. تحقق من النتائج الفعلية؛ تشغيل الدفتر لا يضمن وصول النموذج إلى دقة محددة.


In [ ]:
import json

command = [sys.executable, '-u', str(repo / 'train.py'),
    '--data', str(prepared / 'data.yaml'), '--model', 'yolo11n.pt',
    '--device', '0', '--epochs', '40', '--retry-epochs', '60',
    '--imgsz', '640', '--batch', '16', '--workers', '2', '--output', str(output)]
(output / 'command.json').write_text(json.dumps(command, indent=2))
with (output / 'training.log').open('w') as log:
    process = subprocess.Popen(command, cwd=repo, stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT, text=True, bufsize=1)
    try:
        for line in process.stdout:
            print(line, end='')
            log.write(line)
            log.flush()
        returncode = process.wait()
    except BaseException:
        process.terminate()
        process.wait()
        raise
if returncode:
    raise RuntimeError(f'Training exited with code {returncode}. See {output / "training.log"}')


## النتائج والنموذج المختار

المسار `weights` في التقرير يشير إلى النموذج المختار باستخدام validation. نتيجة test للعرض النهائي، ولا تُستخدم لاختيار النموذج. إذا كانت `meets_validation_targets` تساوي false، ما زالت النتائج دون الأهداف المحددة.


In [ ]:
from IPython.display import display, Image

report = json.loads((output / 'metrics.json').read_text())
print(json.dumps(report, indent=2, ensure_ascii=False))
selected_model = Path(report['weights'])
shutil.copyfile(selected_model, output / 'best_selected.pt')
print('Selected model saved to:', output / 'best_selected.pt')
for name in ['confusion_matrix_normalized.png', 'val_batch0_pred.jpg']:
    plot = output / 'final_test' / name
    if plot.exists():
        display(Image(filename=str(plot), width=900))
